# Fintech Sentiment Engine

End-to-end pipeline: Google Play reviews → BERTopic clustering → Claude strategic brief → Streamlit UI.

| Phase | What | Module |
|-------|------|--------|
| 1 | Data collection — Google Play reviews for 5 apps | `src/data_collection.py` |
| 2 | BERTopic clustering — 9 semantic topic clusters | `src/clustering.py` |
| 3 | Strategic brief — Claude synthesizes a PM memo | `src/brief_generator.py` |
| 4 | Streamlit UI — interactive browser interface | `app/streamlit_app.py` |

**Apps:** Chime · Cash App · Affirm · Klarna · Chase (cobrand baseline)

> Run cells top-to-bottom. Phase 2 loads pre-computed results from disk if available (~60s to re-run).  
> Phase 3 requires `ANTHROPIC_API_KEY` in `.env`.

In [ ]:
import sys, os, warnings, logging
sys.path.insert(0, '..')
warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.WARNING)

import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import Markdown, display

RAW_DIR       = Path('../data/raw')
PROCESSED_DIR = Path('../data/processed')

print('Setup complete')

---
## Phase 1 — Data Collection

Source: Google Play Store via `google-play-scraper`.  
200 reviews per app, newest-first, English only.  
Schema: `date · userName · rating · title · review · app · source · fetched_at`

In [ ]:
from src.data_collection import APP_REGISTRY

print('Target apps:', list(APP_REGISTRY.keys()))
print()

combined_path = RAW_DIR / 'combined_reviews.csv'
if combined_path.exists():
    df_raw = pd.read_csv(combined_path)
    df_raw['date'] = pd.to_datetime(df_raw['date'], errors='coerce')
    print(f'Loaded {len(df_raw):,} reviews from disk')
    print(f'Date range: {df_raw["date"].min().date()} → {df_raw["date"].max().date()}')
    print(f'Sources: {df_raw["source"].value_counts().to_dict()}')
else:
    print('No combined CSV found.')
    print('Run: python src/data_collection.py')
    df_raw = pd.DataFrame()

In [ ]:
if not df_raw.empty:
    stats = df_raw.groupby('app')['rating'].agg(['count', 'mean']).round(2)
    stats.columns = ['reviews', 'avg_rating']
    display(stats.sort_values('avg_rating').style.background_gradient(
        subset=['avg_rating'], cmap='RdYlGn', vmin=1, vmax=5
    ))

    fig, ax = plt.subplots(figsize=(11, 4))
    pivot = df_raw.groupby(['app', 'rating']).size().unstack(fill_value=0)
    pivot.plot(kind='bar', stacked=True, ax=ax, colormap='RdYlGn', width=0.6)
    ax.set_title('Rating distribution by app (Google Play)', fontsize=13)
    ax.set_xlabel('App')
    ax.set_ylabel('Review count')
    ax.legend(title='Stars', bbox_to_anchor=(1.01, 1), loc='upper left')
    ax.tick_params(axis='x', rotation=0)
    plt.tight_layout()
    plt.show()

In [ ]:
if not df_raw.empty:
    display(df_raw[['app', 'rating', 'review']].sample(5, random_state=42))

In [ ]:
# Optional: live scraper smoke test (50 Chime reviews)
# Uncomment to validate the scraper is still working.

# from src.data_collection import fetch_play_reviews
# df_smoke = fetch_play_reviews('chime', how_many=50)
# print(f'Fetched {len(df_smoke)} reviews')
# display(df_smoke[['date', 'rating', 'review']].head())

---
## Phase 2 — BERTopic Clustering

Loads pre-computed results from `data/processed/` if they exist.  
Delete the processed CSVs and re-run this cell to re-cluster from scratch.

**Pipeline:**
```
combined_reviews.csv (850 usable docs)
  → all-MiniLM-L6-v2 embeddings  (850, 384)
  → UMAP  n_components=5, n_neighbors=15, cosine
  → HDBSCAN  min_cluster_size=20  →  13 initial topics
  → reduce_outliers  (26% outlier → 0%)
  → reduce_topics  13 → 10  (semantic merging)
  → consolidate_generic_topics  →  9 meaningful topics  +  94 generic-positive excluded
```

In [ ]:
summary_path = PROCESSED_DIR / 'topic_summary.csv'
reviews_path = PROCESSED_DIR / 'reviews_with_topics.csv'

if summary_path.exists() and reviews_path.exists():
    topic_summary       = pd.read_csv(summary_path)
    reviews_with_topics = pd.read_csv(reviews_path)
    print(f'Loaded from disk: {len(topic_summary)} topic rows, {len(reviews_with_topics):,} reviews')
else:
    print('Running BERTopic pipeline (~60s)...')
    from src.clustering import run_pipeline
    logging.getLogger().setLevel(logging.INFO)
    _, reviews_with_topics, topic_summary, n_generic = run_pipeline()
    logging.getLogger().setLevel(logging.WARNING)
    print(f'Done — {n_generic} generic-positive reviews excluded (topic = -2).')

In [ ]:
from src.brief_generator import TOPIC_LABELS

meaningful = topic_summary[~topic_summary['topic'].isin([-1, -2])].copy()
meaningful['theme'] = meaningful['topic'].map(TOPIC_LABELS)

n_generic_excluded = (
    int(topic_summary.loc[topic_summary['topic'] == -2, 'docs'].iloc[0])
    if -2 in topic_summary['topic'].values else 0
)

print(f'{len(meaningful)} meaningful topics  |  {n_generic_excluded} generic-positive docs excluded\n')

tbl = meaningful[['topic', 'theme', 'docs', 'avg_rating', 'app_breakdown']].copy()
tbl['app_breakdown'] = tbl['app_breakdown'].str.replace('  ', '  ·  ')
tbl = tbl.set_index('topic')

display(tbl.style.background_gradient(subset=['avg_rating'], cmap='RdYlGn', vmin=1, vmax=5))

### Cluster results — current run

850 usable docs · 94 generic-positive excluded · **9 meaningful topics**

| Topic | Theme | Docs | Avg ★ | Dominant app |
|------:|-------|-----:|------:|-------------|
| 0 | Card payments, refunds & merchant friction | 251 | 3.8 | Klarna 31% · Affirm 20% · Chase 19% |
| 1 | App update & login failures | 96 | **1.8** | Chase 45% · Affirm 24% · Klarna 14% |
| 3 | Account closures, dispute denials & credit score damage | 76 | **1.8** | Affirm 36% · Klarna 25% · Cash App 16% |
| 4 | App quality — mixed satisfaction | 69 | 4.1 | Mixed |
| 5 | Chime product experience (SpotMe, early direct deposit) | 62 | 4.4 | Chime 100% |
| 6 | BNPL pay-over-time value proposition | 59 | **4.5** | Klarna 29% · Affirm 22% · Chime 20% |
| 7 | Banking fundamentals (fees, deposits, customer service) | 54 | 3.8 | Chase 44% · Chime 31% |
| 8 | Cash App — fraud, unauthorized charges & deactivation | 47 | 3.6 | Cash App 87% |
| 9 | Affirm — loan terms & interest rate friction | 42 | 3.1 | Affirm 90% · Klarna 7% |

> **Pain clusters:** Topics 1 and 3 both at 1.8★ — the two lowest-rated themes in the corpus. Topic 1 is a Chase-heavy technical failure; Topic 3 is a trust/credit cluster concentrated in Affirm and Klarna.  
> **Positive clusters:** Topic 6 (BNPL pay-over-time, 4.5★) and Topic 5 (Chime, 4.4★) are the highest-rated, driven by users who actively chose these products for specific mechanics.

In [ ]:
plot_data = meaningful.set_index('theme')['avg_rating'].sort_values()
colors = [
    '#d73027' if r < 2.5 else
    '#fc8d59' if r < 3.0 else
    '#fee090' if r < 3.5 else
    '#91cf60' if r < 4.0 else '#1a9850'
    for r in plot_data
]

fig, ax = plt.subplots(figsize=(10, 5))
plot_data.plot(kind='barh', ax=ax, color=colors, edgecolor='white')
ax.axvline(3, color='gray', linestyle='--', linewidth=0.8, alpha=0.7, label='Neutral 3★')
ax.set_xlabel('Avg star rating')
ax.set_xlim(1, 5)
ax.set_title('Average rating by topic cluster', fontsize=13)
ax.legend(fontsize=9)
for i, v in enumerate(plot_data):
    ax.text(v + 0.05, i, f'{v:.1f}', va='center', fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
print('REPRESENTATIVE EXAMPLES PER TOPIC')
print('=' * 80)

for tid, theme in TOPIC_LABELS.items():
    subset = reviews_with_topics[reviews_with_topics['topic'] == tid].dropna(subset=['text'])
    if subset.empty:
        continue
    row = meaningful[meaningful['topic'] == tid]
    avg_r = f"{row['avg_rating'].iloc[0]}★" if not row.empty else ''
    docs_n = int(row['docs'].iloc[0]) if not row.empty else 0
    reps = (
        subset.assign(_len=subset['text'].str.len())
        .sort_values('_len', ascending=False)['text']
        .head(2).tolist()
    )
    print(f'\n── Topic {tid}: {theme}  [{docs_n} docs · {avg_r}]')
    for i, doc in enumerate(reps, 1):
        print(f'  [{i}] {str(doc).replace(chr(10), " ")[:260]}')

In [ ]:
from src.clustering import flag_bnpl_cobrand_themes

audit_summary = meaningful.copy()
audit_summary['rep_docs'] = audit_summary['topic'].apply(
    lambda tid: (
        reviews_with_topics[reviews_with_topics['topic'] == tid]
        .dropna(subset=['text'])
        .assign(_len=lambda d: d['text'].str.len())
        .sort_values('_len', ascending=False)['text']
        .head(3).tolist()
    )
)

flag_bnpl_cobrand_themes(audit_summary)

---
## Phase 3 — Strategic Brief Generation

Calls **`claude-sonnet-4-6`** to synthesize the 9-topic cluster output into a PM-ready strategic memo on BNPL/neobank substitution risk.

**System prompt instructs Claude to:**
- Write as a product analyst presenting to the internal team
- Reason *across* topics — not summarize them one by one
- Cite topic IDs and stats as evidence (`Topic 3, 1.8★, 36% Affirm`)
- Be honest about what the data shows vs. what it doesn't
- Target: 400–600 words · thesis → 2–3 supporting points → recommendation

Requires `ANTHROPIC_API_KEY` in `.env`.

In [ ]:
from src.brief_generator import load_topic_data, _format_topic_context, _USER_FRAMING

summary_df, rep_docs = load_topic_data()
context = _format_topic_context(summary_df, rep_docs)

print(f'Topics passed to Claude: {len(summary_df)}')
print(f'Context length: {len(context):,} chars  (~{len(context)//4:,} tokens)')
print(f'\nFraming:\n  {_USER_FRAMING}')
print('\n--- Context snippet ---')
print(context[:500])

In [ ]:
from src.brief_generator import generate_brief

if not os.getenv('ANTHROPIC_API_KEY'):
    print('ANTHROPIC_API_KEY not set — add it to .env and re-run')
    brief = ''
else:
    print('Generating brief (streaming)...\n')
    print('-' * 80)
    brief = generate_brief(summary=summary_df, rep_docs=rep_docs, stream_to_stdout=True)
    print('-' * 80)
    print(f'\n{len(brief.split())} words')

In [ ]:
if brief:
    display(Markdown(brief))

### Sample brief output — `claude-sonnet-4-6`

---

**The substitution threat is structural, not transactional**

Based on Google Play sentiment analysis across ~850 reviews from Chime, Cash App, Affirm, Klarna, and Chase, the data does not show users explicitly switching from cobrand credit cards to BNPL. What it reveals is something more foundational: two distinct failure modes in incumbent financial products, and a new generation of fintech tools capturing users precisely where those failures are most acute.

**Where the incumbents are losing**

The two sharpest pain clusters are defined by access failures and trust breakdowns, not product features. Topic 1 (app update & login failures, 96 reviews, 1.8★, Chase 45%) shows that Chase's single biggest sentiment problem isn't rates or rewards — it's that users routinely cannot get into the product after a forced update. When the most common complaint is being locked out of your banking app, that's a distribution failure, not a product one. Separately, Topic 3 (account closures, dispute denials & credit score damage, 76 reviews, 1.8★, Affirm 36%) captures a more serious trust erosion: users reporting unexpected shutdowns, BNPL utilization impacting credit scores, and dispute processes that appear to favor merchants over consumers. This cluster is most relevant to cobrand risk — not because it shows direct switching, but because it signals that BNPL's trust proposition is fragile in exactly the ways that matter to credit-card-adjacent users.

**Where the challengers are winning**

Topic 6 (BNPL pay-over-time value proposition, 59 reviews, 4.5★) is the clearest positive signal: users who understand and choose the installment model on their own terms express genuine satisfaction. Representative reviews describe the product as purpose-built for planned purchases — no language about replacing cards, just a different tool for a specific job. Topic 5 (Chime, 62 reviews, 4.4★) and Topic 8 (Cash App, 47 reviews, 3.6★) show neobank satisfaction clustering around specific mechanics — early direct deposit, SpotMe overdraft access — rather than general quality sentiment. These are deliberate primary banking moves, not passive drift.

**What the data can and can't tell us**

The BNPL-vs-cobrand behavioral signals we specifically audited — card switching at checkout, rewards friction comparisons, credit score impact awareness, primary bank replacement — did not form isolated clusters. They appear as scattered sentences within broader topics, meaning the behavior is present in the corpus but not yet a dominant consumer narrative. We cannot claim from this data that users are substituting BNPL for cobrand interchange-generating spend. We *can* claim that BNPL users are satisfied with the value proposition and that the trust/credit damage in Topic 3 creates a real retention risk if left unaddressed.

**Recommendation**

Before scaling this analysis (5k+ reviews, multi-source), the team should resolve one question: is the checkout displacement risk a feature comparison problem or a trust relationship problem? The data points to the latter. Users who moved their primary deposit to Chime or Cash App weren't tempted away by BNPL rewards math — they left over friction and reliability failures that cobrand products haven't closed. If the team is worried about interchange displacement, the signal to watch is Topic 7 (banking fundamentals, 54 reviews, 3.8★, Chase 44%) — that's where a Chase user deciding whether to stay lives.

---
*Generated by `claude-sonnet-4-6` from 9-topic BERTopic cluster output. ~530 words.*

---
## Phase 4 — Streamlit UI

A browser-based interface over the clustering and brief generation pipeline.

**Features:**
- **Topics tab** — cluster table with green/red rating heatmap, doc counts, app breakdown, expandable representative reviews per topic
- **Strategic Brief tab** — streams the Claude brief token-by-token with a live cursor; result persists in session state

**Launch:**
```bash
cd ~/fintech-sentiment-engine
source .venv/bin/activate
streamlit run app/streamlit_app.py
```

Opens at `http://localhost:8501`.